# VCP + CANSLIM-lite Exploration

Run **after** you've populated the cache by either:
- `python -m scripts.screen --date 2024-12-31 --output out/screen.csv`
- `python -m scripts.backtest --start 2020-01-01 --end 2024-12-31 --trades-out out/trades.csv --equity-out out/equity.csv`


In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT / 'src'))
elif (ROOT.parent / 'src').exists():
    os.chdir(ROOT.parent)
    sys.path.insert(0, 'src')

import pandas as pd
import matplotlib.pyplot as plt
import mplfinance as mpf

from vcp.data import load, load_universe
from vcp.indicators import sma
from vcp.screener import screen_at, explain
from vcp.vcp_pattern import detect, _find_swings, _build_contractions

## 1. Top screener results

In [ ]:
AS_OF = '2024-12-31'
results = screen_at(AS_OF)
print(f'{len(results)} matches')
results.head(20)

## 2. Plot top result with K-line + SMAs + contractions

In [ ]:
if results.empty:
    raise RuntimeError('no screener results — change AS_OF or relax filters')

TICKER = results.iloc[0]['ticker']
print(f'Plotting {TICKER}')
df = load(TICKER, '2023-01-01', AS_OF)

df_plot = df.iloc[-180:].copy()
addplots = [
    mpf.make_addplot(sma(df_plot['close'], 50), color='blue'),
    mpf.make_addplot(sma(df_plot['close'], 150), color='orange'),
    mpf.make_addplot(sma(df_plot['close'], 200), color='red'),
]
mpf.plot(
    df_plot.rename(columns=str.capitalize),
    type='candle',
    volume=True,
    addplot=addplots,
    style='yahoo',
    figsize=(12, 7),
    title=f'{TICKER} — last 180 bars',
)

## 3. Diagnostic detail (explain a specific ticker)

In [ ]:
import json
diag = explain(TICKER, AS_OF)
print(json.dumps(diag, indent=2, default=str, ensure_ascii=False))

## 4. Backtest equity curve + R-multiple histogram

In [ ]:
equity_path = Path('out/equity.csv')
trades_path = Path('out/trades.csv')
if not equity_path.exists():
    print('Run scripts/backtest.py first to populate out/equity.csv and out/trades.csv')
else:
    eq = pd.read_csv(equity_path, index_col=0, parse_dates=True).iloc[:, 0]
    fig, ax = plt.subplots(figsize=(12, 4))
    eq.plot(ax=ax, title='Equity curve')
    ax.grid(True)
    plt.show()
    if trades_path.exists():
        tr = pd.read_csv(trades_path, parse_dates=['entry_date', 'exit_date'])
        fig, ax = plt.subplots(figsize=(10, 4))
        tr['return_pct'].hist(bins=30, ax=ax)
        ax.set_title('Trade return distribution')
        ax.set_xlabel('return %')
        plt.show()
        print(tr.describe())